In [ ]:
!kaggle datasets download -d pankrzysiu/cifar10-python --force

In [ ]:
!unzip -o -q cifar10-python.zip -d ./cifar10_data

In [ ]:
import pickle
import numpy as np
import os
import cv2

In [ ]:
def unpickle(file):
    with open(file, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')
    return data_dict

data_dir = './cifar10_data/cifar-10-batches-py/'
x_train_list = []
y_train_list = []
for i in range(1, 6):
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    batch = unpickle(batch_path)
    x_train_list.append(batch[b'data'])
    y_train_list.append(batch[b'labels'])
x_train = np.concatenate(x_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
test_batch = unpickle(os.path.join(data_dir, 'test_batch'))
x_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])
meta_batch = unpickle(os.path.join(data_dir, 'batches.meta'))
classes = [b.decode('utf-8') for b in meta_batch[b'label_names']]

In [ ]:
def compute_output_size(w, k, s, p):
    return ((w - k + 2 * p) // s) + 1

size_no_pad = compute_output_size(32, 3, 1, 0)
size_pad_1 = compute_output_size(32, 3, 1, 1)
print(f"Output size without padding = {size_no_pad} x {size_no_pad}")
print(f"Output size with padding = {size_pad_1} x {size_pad_1}")

In [ ]:
def convolve2d(image, kernel, padding=0):
    if padding > 0:
        image = np.pad(image, padding, mode='constant', constant_values=0)
    img_h, img_w = image.shape
    kh, kw = kernel.shape
    out_h = img_h - kh + 1
    out_w = img_w - kw + 1
    output = np.zeros((out_h, out_w), dtype=np.int32)
    for r in range(out_h):
        for c in range(out_w):
            output[r, c] = np.sum(image[r:r+kh, c:c+kw] * kernel)
    return output

In [ ]:
input_patch = np.array([
    [61, 45, 48, 57, 78, 96],
    [19, 0, 10, 32, 59, 89],
    [24, 9, 31, 57, 80, 99],
    [26, 24, 61, 72, 79, 82],
    [36, 38, 73, 89, 86, 85],
    [53, 59, 80, 91, 97, 93]
])
filter_kernel = np.array([
    [0, 1, 1],
    [-1, -1, 0],
    [1, -1, 0]
])
output_no_pad = convolve2d(input_patch, filter_kernel, padding=0)
output_pad_1 = convolve2d(input_patch, filter_kernel, padding=1)

print("Input Patch:")
print(input_patch)
print("\nFilter:")
print(filter_kernel)
print("\nOutput size without padding: 30 x 30")
print("Output size with padding: 32 x 32")
print("\nOutput without padding:")
print(output_no_pad)
print(f"Shape: {output_no_pad.shape}")
print("\nOutput with padding:")
print(output_pad_1)
print(f"Shape: {output_pad_1.shape}")

In [ ]:
single_image_flat = x_train[0]
single_image_rgb = single_image_flat.reshape(3, 32, 32).transpose(1, 2, 0)
single_image_gray = cv2.cvtColor(single_image_rgb, cv2.COLOR_RGB2GRAY)
extracted_patch = single_image_gray[10:16, 10:16]

out_no_pad_cifar = convolve2d(extracted_patch, filter_kernel, padding=0)
out_pad_1_cifar = convolve2d(extracted_patch, filter_kernel, padding=1)

print("Extracted CIFAR-10 Patch:")
print(extracted_patch)
print("\nFilter:")
print(filter_kernel)
print("\nOutput size without padding: 30 x 30")
print("Output size with padding: 32 x 32")
print("\nOutput without padding:")
print(out_no_pad_cifar)
print(f"Shape: {out_no_pad_cifar.shape}")
print("\nOutput with padding:")
print(out_pad_1_cifar)
print(f"Shape: {out_pad_1_cifar.shape}")